# Exploración de las ventas de 2026 (resuelto)

Copia del cuaderno `ejemplos/07_cuaderno_exploracion.ipynb` con el apartado **Tu turno** resuelto y un
apartado nuevo que mide cuánto tarda pandas frente a las listas de diccionarios del Módulo 4. El cuaderno
original se deja sin tocar.

Antes de dar nada por bueno: **Reiniciar y ejecutar todo**.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.options.display.float_format = "{:,.2f}".format

## 1. Cargar y mirar

In [ ]:
ventas = pd.read_csv("../datos/ventas_2026.csv", parse_dates=["fecha"])
ventas.head()

In [ ]:
ventas.info()

In [ ]:
ventas.describe()

La última expresión de una celda se muestra sola. Para ver varias cosas en la misma celda, usa `print` o `display`.

In [ ]:
print("Vendedores:", sorted(ventas["vendedor"].unique()))
ventas["region"].value_counts()

## 2. Columnas calculadas

In [ ]:
ventas["importe"] = ventas["unidades"] * ventas["precio"]
ventas["mes"] = ventas["fecha"].dt.strftime("%Y-%m")
ventas[["fecha", "vendedor", "producto", "unidades", "precio", "importe", "mes"]].head()

## 3. Preguntas

### ¿Quién vende más?

In [ ]:
por_vendedor = ventas.groupby("vendedor")["importe"].sum().sort_values(ascending=False)
por_vendedor

In [ ]:
figura, ejes = plt.subplots(figsize=(8, 4.5))
ejes.bar(por_vendedor.index, por_vendedor.values, color="#2b6cb0")
ejes.set_title("Importe por vendedor")
ejes.set_ylabel("Importe (€)")
figura.tight_layout()

### ¿Cómo evolucionan las ventas?

In [ ]:
mensual = ventas.groupby("mes")["importe"].sum()
mensual.plot(marker="o", title="Evolución mensual", ylabel="Importe (€)", figsize=(8, 4.5), grid=True);

### ¿Qué se vende en cada región?

In [ ]:
ventas.pivot_table(index="region", columns="categoria", values="importe", aggfunc="sum", fill_value=0)

### ¿Se cumplen los objetivos?

In [ ]:
objetivos = pd.read_csv("../datos/objetivos.csv")
ranking = por_vendedor.reset_index().merge(objetivos, on="vendedor", how="left")
ranking["cumplimiento"] = (ranking["importe"] / ranking["objetivo"] * 100).round(1)
ranking["cumplido"] = ranking["importe"] >= ranking["objetivo"]
ranking

## 4. Tu turno

Responde en celdas nuevas:

1. ¿Cuál es el producto con más unidades vendidas? ¿Y el que más importe genera? ¿Coinciden?
2. ¿Qué día de la semana se vende más? Pista: `ventas["fecha"].dt.day_name()`.
3. Dibuja un histograma del importe por venta. ¿Qué forma tiene?
4. ¿Qué vendedor tiene el ticket medio más alto en la categoría Software?

### 1. Producto con más unidades y producto con más importe

In [ ]:
unidades_por_producto = ventas.groupby("producto")["unidades"].sum().sort_values(ascending=False)
importe_por_producto = ventas.groupby("producto")["importe"].sum().sort_values(ascending=False)
print("Más unidades:", unidades_por_producto.idxmax(), f"({unidades_por_producto.max()} unidades)")
print("Más importe: ", importe_por_producto.idxmax(), f"({importe_por_producto.max():,.2f} €)")
pd.DataFrame({"unidades": unidades_por_producto, "importe": importe_por_producto}).sort_values("importe", ascending=False)

No coinciden: el producto que más unidades vende es barato (se vende en cantidad), mientras que el que más
importe genera es caro y se vende en pocas unidades. Para el negocio, el importe es lo que cuenta.

### 2. Día de la semana con más ventas

In [ ]:
DIAS = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
dia = ventas["fecha"].dt.day_of_week.map(dict(enumerate(DIAS)))
por_dia = ventas.groupby(dia).agg(ventas=("importe", "size"), importe=("importe", "sum")).reindex(DIAS).dropna()
por_dia

In [ ]:
print("Más ventas:", por_dia["ventas"].idxmax(), "| Más importe:", por_dia["importe"].idxmax())

`dt.day_name()` devuelve los nombres en inglés, así que se traducen con `day_of_week` (0 es el lunes) y se
reordenan con `reindex` para que salgan en el orden de la semana y no en orden alfabético.

### 3. Histograma del importe por venta

In [ ]:
figura, ejes = plt.subplots(figsize=(8, 4.5))
ejes.hist(ventas["importe"], bins=30, color="#2b6cb0", edgecolor="white")
ejes.set_title("Importe por venta")
ejes.set_xlabel("Importe (€)")
ejes.set_ylabel("Número de ventas")
figura.tight_layout()

In [ ]:
ventas["importe"].describe()

Es muy asimétrica hacia la derecha: la mayoría de las ventas son pequeñas (accesorios y licencias) y unas
pocas ventas de equipos caros forman una cola larga. Por eso la media queda bastante por encima de la mediana.

### 4. Ticket medio más alto en Software

In [ ]:
software = ventas[ventas["categoria"] == "Software"]
ticket_software = software.groupby("vendedor")["importe"].mean().round(2).sort_values(ascending=False)
print("Ticket medio más alto en Software:", ticket_software.idxmax())
ticket_software

## 5. Conclusiones

Escribe aquí, en dos o tres frases, lo que has encontrado. Un análisis sin conclusiones es solo una colección de tablas.

- **Ada** es quien más factura y una de las dos personas que superan su objetivo (la otra es Linus), aunque
  Grace hace más ventas: el ticket medio de Ada es más alto.
- El mes más fuerte es **septiembre** y el más flojo **febrero**; la evolución mensual es irregular, con
  subidas y bajadas de más del 20 % casi cada mes.
- Las unidades las mueve el producto barato (**Antivirus**), pero el importe lo traen los **equipos**: unas
  pocas ventas grandes explican buena parte del total, como se ve en el histograma.

## 6. ¿Cuánto más rápido es pandas?

Se compara `importe_por` de la Práctica 4 (bucles sobre una lista de diccionarios) con su equivalente en
pandas (`groupby`), con las 320 ventas repetidas mil veces: 320 000 filas. `agrupar_por` del Ejercicio 6 del
Módulo 4 solo forma los grupos; `importe_por` hace además la suma, que es lo que hace `groupby` aquí.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..", "..", "modulo_04_colecciones", "ejercicios").resolve()))
import m4_practica_ventas as p4

grandes = pd.concat([ventas] * 1000, ignore_index=True)
lista = grandes[["fecha", "vendedor", "region", "producto", "categoria", "unidades", "precio"]].assign(
    fecha=grandes["fecha"].dt.strftime("%Y-%m-%d")
).to_dict("records")
len(grandes), len(lista)

In [ ]:
# Las dos versiones dan el mismo resultado
con_pandas = grandes.groupby("region")["importe"].sum().round(2).to_dict()
con_listas = p4.importe_por(lista, "region")
{region: (con_pandas[region], con_listas[region]) for region in con_listas}

In [ ]:
%timeit p4.importe_por(lista, "region")

In [ ]:
%timeit grandes.groupby("region")["importe"].sum()

Con 320 000 filas la versión de pandas es **más de diez veces más rápida** (en mi equipo, unos 20 ms frente
a unos 280 ms; el número exacto depende del ordenador): el bucle lo hace NumPy en
código compilado, sin crear un objeto de Python por cada venta. Con las 20 ventas del Módulo 4 la diferencia
no se nota (incluso puede ganar la lista, porque pandas tiene un coste fijo de preparación); con datos de
verdad es lo que decide qué herramienta usar.